In [1]:
import os
import requests
import pandas as pd
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

In [2]:
from google.cloud import bigquery
import pandas as pd

# Inicializa el cliente de BigQuery
client = bigquery.Client(project='dataton-2024-team-01-cofares')

# Ejecuta la consulta y convierte los datos en un DataFrame de Pandas desde BigQuery datos_no_descriptions_eans
query = "SELECT * FROM `dataton-2024-team-01-cofares.datos_cofares.img_data_temp`"
df = client.query(query).to_dataframe()
print(df.head())

# Leer el archivo df_img_description.parquet para utilizarlo como df
#df = pd.read_parquet("df_img_description.parquet")
#print(df.head())

/Users/gabrielnoguera/Documents/DataHub/app-flask/cofaresapp/.venv/lib/python3.11/site-packages/google/cloud/bigquery/table.py:1727: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(
/Users/gabrielnoguera/Documents/DataHub/app-flask/cofaresapp/.venv/lib/python3.11/site-packages/google/cloud/bigquery/_pandas_helpers.py:207: UserWarning: Unable to determine Arrow type for field 'ml_generate_text_result'.
  warnings.warn(


                                                 uri forma color  \
0  gs://dataton-2024-team-01-cofares-datastore/im...  None  None   
1  gs://dataton-2024-team-01-cofares-datastore/im...  None  None   
2  gs://dataton-2024-team-01-cofares-datastore/im...  None  None   
3  gs://dataton-2024-team-01-cofares-datastore/im...  None  None   
4  gs://dataton-2024-team-01-cofares-datastore/im...  None  None   

  descripcion_visual empaque zona_de_aplicacion  \
0               None    None               None   
1               None    None               None   
2               None    None               None   
3               None    None               None   
4               None    None               None   

                             ml_generate_text_result  
0  {'candidates': [{'avg_logprobs': -0.1555664457...  
1  {'candidates': [{'avg_logprobs': -0.1555664457...  
2  {'candidates': [{'avg_logprobs': -0.1555664457...  
3  {'candidates': [{'avg_logprobs': -0.1555664457...  
4  {'cand

In [4]:
import pandas as pd

# Supongamos que df es tu DataFrame y ya contiene la columna "ml_generate_text_result"
# Extraemos los datos del JSON
def extraer_datos(data):
    if data is None or 'candidates' not in data or not data['candidates']:
        return {  # Retornar un diccionario vacío si data es None o no tiene la estructura esperada
            'forma': None,
            'color': None,
            'descripcion_visual': None,
            'empaque': None,
            'zona_de_aplicacion': None
        }
    
    try:
        # Extraer el string que representa el JSON
        forma = data['candidates'][0]['content']['parts'][0]['text']
        forma_dict = eval(forma)  # Convertir el string a un diccionario
        
        # Retornar un diccionario con los datos extraídos
        return {
            'forma': forma_dict.get('forma'),
            'color': forma_dict.get('color'),
            'descripcion_visual': forma_dict.get('descripcion_visual'),
            'empaque': forma_dict.get('empaque'),
            'zona_de_aplicacion': forma_dict.get('zona_de_aplicacion')
        }
    except (IndexError, KeyError, SyntaxError) as e:
        return {  # Retornar un diccionario vacío en caso de error
            'forma': None,
            'color': None,
            'descripcion_visual': None,
            'empaque': None,
            'zona_de_aplicacion': None
        }

# Aplicar la función a la columna del DataFrame y expandir el resultado en nuevas columnas
df[['forma', 'color', 'descripcion_visual', 'empaque', 'zona_de_aplicacion']] = df['ml_generate_text_result'].apply(extraer_datos).apply(pd.Series)

# Mostrar el DataFrame con los datos extraídos
print(df[['ml_generate_text_result', 'forma', 'color', 'descripcion_visual', 'empaque', 'zona_de_aplicacion']])

                                 ml_generate_text_result     forma  \
0      {'candidates': [{'avg_logprobs': -0.1555664457...  Cilindro   
1      {'candidates': [{'avg_logprobs': -0.1555664457...  Cilindro   
2      {'candidates': [{'avg_logprobs': -0.1555664457...  Cilindro   
3      {'candidates': [{'avg_logprobs': -0.1555664457...  Cilindro   
4      {'candidates': [{'avg_logprobs': -0.1555664457...  Cilindro   
...                                                  ...       ...   
23431  {'candidates': [{'avg_logprobs': -0.1555664457...  Cilindro   
23432  {'candidates': [{'avg_logprobs': -0.1555664457...  Cilindro   
23433  {'candidates': [{'avg_logprobs': -0.1555664457...  Cilindro   
23434  {'candidates': [{'avg_logprobs': -0.1555664457...  Cilindro   
23435  {'candidates': [{'avg_logprobs': -0.1555664457...  Cilindro   

              color                                 descripcion_visual  \
0      Azul, blanco  Un tubo de plástico con una etiqueta azul y bl...   
1      Azul